# High-Throughput LLM KV Cache Paging & Pinned Memory Offloading

## 1. The Serving Bottleneck & Virtual Memory Paging
During autoregressive text generation, the Key-Value (KV) cache grows linearly with sequence length $S$. In naive LLM serving engines, contiguous memory buffers are pre-allocated for the maximum context window (e.g., 8K or 32K tokens). This causes massive **internal fragmentation** (reserved memory never used by short queries) and **external fragmentation** (inability to pack concurrent requests), wasting up to 60-80% of GPU VRAM!

### PagedAttention Architecture
Inspired by virtual memory in operating systems, **PagedAttention** (vLLM) divides the continuous KV cache of a sequence into fixed-size physical memory blocks (typically 16 or 32 tokens).
- **Logical Blocks**: Sequential token chunks generated by the query.
- **Physical Blocks**: Distributed non-contiguously across GPU VRAM.
- **Block Table**: Maps logical block indices to physical block IDs.

$$\text{Memory Utilization} \approx \frac{\text{Active Tokens}}{\text{Allocated Blocks} \times \text{Block Size}} > 96\%$$

### Host Offloading via Pinned (Page-Locked) Memory
When total concurrent requests exceed GPU memory, cold or long-idle KV blocks are streamed to Host CPU RAM via asynchronous DMA transfers using **CUDA Pinned Memory** (`cudaMemcpyAsync`), preventing Out-Of-Memory (OOM) crashes.

In [ ]:
import numpy as np

class SimulatedPagedKVCache:
    """
    Demonstrates Block Table virtual memory mapping and fragmentation reduction for KV caching.
    """
    def __init__(self, num_gpu_blocks=32, block_size=16, hidden_dim=64):
        self.num_gpu_blocks = num_gpu_blocks
        self.block_size = block_size
        self.hidden_dim = hidden_dim
        
        # Pre-allocated physical pool on GPU [num_blocks, block_size, hidden_dim]
        self.gpu_memory_pool = np.zeros((num_gpu_blocks, block_size, hidden_dim), dtype=np.float32)
        self.free_blocks = list(range(num_gpu_blocks))
        self.block_tables = {} # req_id -> list of physical block indices

    def allocate_for_request(self, req_id, num_tokens):
        """Allocates non-contiguous physical blocks for incoming request tokens."""
        needed_blocks = int(np.ceil(num_tokens / self.block_size))
        assert len(self.free_blocks) >= needed_blocks, "GPU KV Memory Exhausted!"
        
        allocated = [self.free_blocks.pop(0) for _ in range(needed_blocks)]
        self.block_tables[req_id] = allocated
        return allocated

    def write_kv_token(self, req_id, token_idx, kv_vector):
        """Translates logical token index to physical block and offset."""
        block_table = self.block_tables[req_id]
        logical_block_idx = token_idx // self.block_size
        offset_in_block = token_idx % self.block_size
        physical_block_id = block_table[logical_block_idx]
        
        self.gpu_memory_pool[physical_block_id, offset_in_block] = kv_vector

    def release_request(self, req_id):
        """Frees blocks back to the pool."""
        blocks = self.block_tables.pop(req_id, [])
        self.free_blocks.extend(blocks)

# Simulation
cache_manager = SimulatedPagedKVCache(num_gpu_blocks=10, block_size=16, hidden_dim=8)

# Request A has 35 tokens -> requires ceil(35/16) = 3 blocks
blocks_A = cache_manager.allocate_for_request("req_A", 35)
# Request B has 12 tokens -> requires ceil(12/16) = 1 block
blocks_B = cache_manager.allocate_for_request("req_B", 12)

print(f"Request A (35 tokens) mapped to physical blocks: {blocks_A}")
print(f"Request B (12 tokens) mapped to physical blocks: {blocks_B}")
print(f"Remaining Free Physical Blocks: {len(cache_manager.free_blocks)} / 10")

# Write token 20 for Request A (belongs to block index 1, offset 4)
dummy_kv = np.ones(8) * 42.0
cache_manager.write_kv_token("req_A", token_idx=20, kv_vector=dummy_kv)

target_phys_block = blocks_A[20 // 16]
assert np.all(cache_manager.gpu_memory_pool[target_phys_block, 20 % 16] == 42.0)
print(f"Logical token 20 successfully stored into physical block {target_phys_block} offset {20 % 16}!")

## 2. Host Offloading & Asynchronous Pinned Transfer Benchmark Simulation

In [ ]:
def kv_cache_size_bytes(seq_len, num_layers=32, num_heads=32, head_dim=128, precision_bytes=2):
    """
    KV Cache per token formula:
    2 (Key + Value) * num_layers * num_heads * head_dim * precision_bytes
    """
    bytes_per_token = 2 * num_layers * num_heads * head_dim * precision_bytes
    total_bytes = seq_len * bytes_per_token
    return total_bytes / (1024**2) # in MiB

seq_lengths = [512, 2048, 8192, 32768]
print("KV Cache Size per Single Request (LLaMA-3 8B / 32 layers / 32 heads / FP16):")
print("-" * 65)
for s in seq_lengths:
    size_mb = kv_cache_size_bytes(s)
    print(f"  Sequence: {s:5d} tokens | KV Cache: {size_mb:8.1f} MiB ({size_mb / 1024:.2f} GiB)")

## 3. High-Yield Interview Questions & Trade-offs

### Q1: Why is Standard Memory (Pageable) significantly slower than Pinned Memory for GPU-to-CPU offloading?
**Answer:** Standard host memory is managed by OS virtual memory and can be paged out to disk at any time. When moving data between GPU and pageable host memory, the CUDA driver must first allocate a temporary pinned host staging buffer, copy the data synchronously to that buffer via CPU, and then initiate the PCIe transfer. **Pinned (page-locked) memory** is locked in physical RAM, allowing the GPU's Direct Memory Access (DMA) engine to read/write host RAM asynchronously without CPU intervention, achieving maximum PCIe bandwidth ($approx 32-64$ GB/s on PCIe Gen4/5).

### Q2: What is the optimal block size in PagedAttention and what are the trade-offs?
**Answer:** The most common block sizes are 16 or 32 tokens:
- **Smaller block size (e.g. 8 or 16)** minimizes internal fragmentation in the last block of each sequence, yielding >96% memory utilization. However, it increases the size of the block table and overhead of kernel lookup dispatching.
- **Larger block size (e.g. 64 or 128)** enables coalesced memory loads and better GPU hardware warp occupancy during attention kernel execution, but increases waste when requests end midway through a block.